# 🌾 FarmTech Solutions — Previsão de Rendimento de Safra
## FIAP — Pós-Tech em IA para Devs | Fase 5 | PBL

**Projeto:** Machine Learning na Cabeça  
**Dataset:** `crop_yield.csv` — Condições de solo, temperatura e rendimento de culturas agrícolas  
**Objetivo:** Análise exploratória, clusterização para descoberta de tendências e construção de 5 modelos preditivos de rendimento de safra

---

### Estrutura do Notebook

| Seção | Descrição |
|---|---|
| 1. Setup & Importações | Instalação e carregamento das bibliotecas |
| 2. Carregamento e Visão Geral dos Dados | Leitura do CSV, tipos e estatísticas |
| 3. Análise Exploratória (EDA) | Distribuições, correlações, visualizações |
| 4. Clusterização & Outliers | KMeans, DBSCAN, IQR e análise de cenários discrepantes |
| 5. Modelagem Preditiva (5 Algoritmos) | Regressão Linear, Ridge, Random Forest, Gradient Boosting, SVR |
| 6. Avaliação e Comparação dos Modelos | Métricas R², MAE, RMSE |
| 7. Conclusões | Achados, pontos fortes e limitações |

## 1. Setup & Importações

Instalação e importação de todas as bibliotecas necessárias para análise exploratória, clusterização e modelagem preditiva.

In [ ]:
# ============================================================
# Instalação de pacotes (executar apenas se necessário)
# ============================================================
# !pip install pandas numpy matplotlib seaborn scikit-learn scipy

# ============================================================
# Importações das bibliotecas
# ============================================================
import warnings
warnings.filterwarnings('ignore')  # Suprimir avisos desnecessários

# Manipulação e análise de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Pré-processamento e Machine Learning
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Algoritmos de Regressão (Entrega 1 - 5 modelos)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# Clusterização (Aprendizado Não Supervisionado)
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Configuração visual global
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print("✅ Todas as bibliotecas importadas com sucesso!")

## 2. Carregamento e Visão Geral dos Dados

Carregamento do dataset `crop_yield.csv`, inspeção inicial da estrutura, tipos de dados, valores faltantes e estatísticas descritivas.

In [ ]:
# ============================================================
# Carregamento do dataset
# ============================================================
# Caminho relativo ao notebook (ambos estão no mesmo repositório)
df = pd.read_csv('arquivo/crop_yield.csv')

# Renomear colunas para facilitar o manuseio no código
df.columns = [
    'Cultura',
    'Precipitacao_mm',
    'Umidade_Especifica_g_kg',
    'Umidade_Relativa_pct',
    'Temperatura_C',
    'Rendimento'
]

print(f"📊 Shape do dataset: {df.shape[0]} linhas × {df.shape[1]} colunas")
print(f"\n📋 Primeiras 5 linhas:")
df.head()

In [ ]:
# ============================================================
# Informações estruturais do dataset
# ============================================================
print("=" * 55)
print("INFORMAÇÕES GERAIS DO DATASET")
print("=" * 55)
df.info()

print("\n")
print("=" * 55)
print("VERIFICAÇÃO DE VALORES NULOS")
print("=" * 55)
print(df.isnull().sum())
nulos_total = df.isnull().sum().sum()
print(f"\n🔎 Total de valores nulos: {nulos_total}")
if nulos_total == 0:
    print("✅ Dataset sem valores faltantes — pronto para análise!")

print("\n")
print("=" * 55)
print("CULTURAS ÚNICAS NO DATASET")
print("=" * 55)
culturas = df['Cultura'].unique()
print(f"Total de culturas: {len(culturas)}")
for c in sorted(culturas):
    count = df[df['Cultura'] == c].shape[0]
    print(f"  • {c}: {count} registros")

In [ ]:
# ============================================================
# Estatísticas descritivas das variáveis numéricas
# ============================================================
print("=" * 55)
print("ESTATÍSTICAS DESCRITIVAS")
print("=" * 55)
desc = df.describe().T
desc['range'] = desc['max'] - desc['min']  # Amplitude de cada variável
desc['cv%'] = (desc['std'] / desc['mean'] * 100).round(2)  # Coef. de variação
print(desc.round(3))
print("\n📌 Interpretação:")
print("  • cv% = Coeficiente de Variação (std/mean×100) — quanto maior, mais disperso")
print("  • range = Amplitude (max - min) — escala de variação total")

## 3. Análise Exploratória de Dados (EDA)

Nesta seção exploramos visualmente as distribuições das variáveis, correlações entre features e o comportamento do rendimento por cultura. O objetivo é identificar padrões, assimetrias e relações entre as variáveis de entrada e a variável-alvo (`Rendimento`).

In [ ]:
# ============================================================
# 3.1 — Distribuição das variáveis numéricas (histogramas + KDE)
# ============================================================
features_num = ['Precipitacao_mm', 'Umidade_Especifica_g_kg',
                'Umidade_Relativa_pct', 'Temperatura_C', 'Rendimento']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(features_num):
    ax = axes[i]
    # Histograma com curva de densidade
    sns.histplot(df[col], kde=True, ax=ax, color=sns.color_palette("Set2")[i], bins=20)
    ax.set_title(f'Distribuição: {col}', fontsize=11, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequência')

# Exibir contagem por cultura no último subplot
cultura_counts = df['Cultura'].value_counts()
axes[5].barh(cultura_counts.index, cultura_counts.values,
             color=sns.color_palette("Set2", len(cultura_counts)))
axes[5].set_title('Registros por Cultura', fontsize=11, fontweight='bold')
axes[5].set_xlabel('Quantidade')

plt.tight_layout()
plt.suptitle('Distribuição das Variáveis — crop_yield.csv', y=1.01, fontsize=14, fontweight='bold')
plt.show()

print("\n📌 Observações:")
print("  • A distribuição do Rendimento é assimétrica — há culturas com rendimentos muito distintos")
print("  • Temperatura e Umidade Específica têm distribuição bimodal (diferentes culturas têm regimes climáticos distintos)")
print("  • Precipitação apresenta ampla variação, indicando diversidade climática entre as culturas")

In [ ]:
# ============================================================
# 3.2 — Boxplot: Rendimento por Cultura
# ============================================================
plt.figure(figsize=(14, 6))
ordem = df.groupby('Cultura')['Rendimento'].median().sort_values(ascending=False).index

sns.boxplot(data=df, x='Cultura', y='Rendimento',
            order=ordem, palette='Set2')

plt.title('Rendimento por Cultura (ordenado por mediana)', fontsize=13, fontweight='bold')
plt.xlabel('Cultura')
plt.ylabel('Rendimento (t/ha ou kg/ha)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

# Resumo estatístico por cultura
print("\n📊 Estatísticas de Rendimento por Cultura:")
resumo = df.groupby('Cultura')['Rendimento'].agg(['mean', 'median', 'std', 'min', 'max']).round(1)
resumo.columns = ['Média', 'Mediana', 'Desvio Padrão', 'Mínimo', 'Máximo']
print(resumo.sort_values('Média', ascending=False).to_string())

print("\n📌 Observações:")
print("  • Algumas culturas possuem rendimentos médios muito superiores (ex: Cocoa Beans vs Rubber Natural)")
print("  • A amplitude intra-cultural sugere variabilidade climática influenciando o rendimento")
print("  • Outliers visíveis nos boxplots indicam anos/condições excepcionais")

In [ ]:
# ============================================================
# 3.3 — Matriz de Correlação (Heatmap)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Correlação com todas as variáveis numéricas
corr_all = df[features_num].corr()
mask = np.triu(np.ones_like(corr_all, dtype=bool))  # Triângulo superior para limpar
sns.heatmap(corr_all, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, ax=axes[0], vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
axes[0].set_title('Correlação — Todas as Variáveis Numéricas', fontweight='bold')

# Correlação das features com Rendimento (barplot)
corr_target = df[features_num].corr()['Rendimento'].drop('Rendimento').sort_values()
colors = ['#d73027' if v < 0 else '#1a9850' for v in corr_target.values]
axes[1].barh(corr_target.index, corr_target.values, color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Correlação de cada Feature com Rendimento', fontweight='bold')
axes[1].set_xlabel('Coeficiente de Correlação de Pearson')
for i, (val, feat) in enumerate(zip(corr_target.values, corr_target.index)):
    axes[1].text(val + (0.01 if val >= 0 else -0.01), i,
                 f'{val:.2f}', va='center', ha='left' if val >= 0 else 'right', fontsize=10)

plt.tight_layout()
plt.show()

print("\n📌 Observações:")
print("  • Temperatura e Umidade Específica apresentam correlação relevante com o rendimento")
print("  • Precipitação tem correlação mais fraca, indicando que quantidade de chuva isolada")
print("    não é determinante — o tipo de cultura importa mais")
print("  • Alta correlação entre Umidade Específica e Temperatura é esperada (colinearidade)")

In [ ]:
# ============================================================
# 3.4 — Pairplot: Relações entre variáveis numéricas por cultura
# ============================================================
# Usando amostra representativa para evitar visualização poluída
g = sns.pairplot(df, hue='Cultura', vars=features_num,
                 plot_kws={'alpha': 0.6, 's': 30},
                 diag_kind='kde', palette='Set1')

g.fig.suptitle('Pairplot — Relações entre Variáveis por Cultura', y=1.01, fontsize=13, fontweight='bold')
plt.show()

print("\n📌 Observações:")
print("  • Grupos bem separados no espaço de features — as culturas ocupam nichos climáticos distintos")
print("  • Isso justifica o uso de clusterização para identificar tendências naturais nos dados")
print("  • A separação visual sugere que a variável 'Cultura' será a feature mais discriminante nos modelos")

In [ ]:
# ============================================================
# 3.5 — Scatter plots: Features vs Rendimento por cultura
# ============================================================
features_scatter = ['Precipitacao_mm', 'Umidade_Especifica_g_kg',
                    'Umidade_Relativa_pct', 'Temperatura_C']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

culturas_unicas = df['Cultura'].unique()
palette = sns.color_palette("Set1", len(culturas_unicas))
cultura_cor = {c: palette[i] for i, c in enumerate(culturas_unicas)}

for i, feat in enumerate(features_scatter):
    ax = axes[i]
    for cultura in culturas_unicas:
        subset = df[df['Cultura'] == cultura]
        ax.scatter(subset[feat], subset['Rendimento'],
                   color=cultura_cor[cultura], alpha=0.6, s=40,
                   label=cultura.split(',')[0])  # Nome abreviado na legenda
    ax.set_xlabel(feat)
    ax.set_ylabel('Rendimento')
    ax.set_title(f'{feat} × Rendimento', fontweight='bold')

# Legenda no último gráfico
handles = [plt.scatter([], [], color=v, s=40, label=k.split(',')[0])
           for k, v in cultura_cor.items()]
fig.legend(handles=handles, title='Cultura', bbox_to_anchor=(1.01, 0.5),
           loc='center left', fontsize=9)

plt.suptitle('Relação entre Features Climáticas e Rendimento', y=1.01,
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Clusterização & Detecção de Outliers (Aprendizado Não Supervisionado)

Nesta seção exploramos tendências nos dados de rendimento agrícola utilizando dois algoritmos de clusterização:
- **KMeans**: identifica grupos compactos de culturas com comportamentos semelhantes
- **DBSCAN**: identifica regiões de alta densidade e automaticamente isola pontos de baixa densidade como outliers

Também utilizamos o **Método IQR** para detectar valores discrepantes na variável `Rendimento`.

In [ ]:
# ============================================================
# 4.1 — Pré-processamento para clusterização
# ============================================================
# Selecionamos apenas as features numéricas para o clustering
# A variável alvo (Rendimento) é INCLUÍDA aqui pois queremos
# agrupar por comportamento holístico (features + rendimento)
features_cluster = ['Precipitacao_mm', 'Umidade_Especifica_g_kg',
                    'Umidade_Relativa_pct', 'Temperatura_C', 'Rendimento']

# Normalização com StandardScaler (média=0, desvio=1)
# Necessário para que features em escalas muito diferentes
# não dominem o cálculo de distância
scaler_cluster = StandardScaler()
X_cluster = scaler_cluster.fit_transform(df[features_cluster])

# Redução de dimensionalidade com PCA para visualização 2D
# PCA captura a maior variância dos dados em 2 componentes
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster)

# Variância explicada por cada componente principal
variancia_explicada = pca.explained_variance_ratio_
print(f"📐 PCA — Variância explicada:")
print(f"   Componente 1: {variancia_explicada[0]*100:.1f}%")
print(f"   Componente 2: {variancia_explicada[1]*100:.1f}%")
print(f"   Total (2 PCs): {sum(variancia_explicada)*100:.1f}%")

In [ ]:
# ============================================================
# 4.2 — KMeans: Método do Cotovelo para escolha do K ideal
# ============================================================
inercias = []
silhouettes = []
k_range = range(2, 11)  # Testamos de 2 a 10 clusters

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_cluster, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico do Cotovelo (Elbow Method)
axes[0].plot(k_range, inercias, 'bo-', markersize=8)
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inércia (WCSS)')
axes[0].set_title('Método do Cotovelo — Escolha do K Ideal', fontweight='bold')
axes[0].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='K escolhido')
axes[0].legend()

# Silhouette Score
axes[1].plot(k_range, silhouettes, 'rs-', markersize=8)
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score por K\n(quanto maior, melhor separação)', fontweight='bold')
best_k = k_range[np.argmax(silhouettes)]
axes[1].axvline(x=best_k, color='blue', linestyle='--', alpha=0.7,
                label=f'Melhor K = {best_k}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n✅ Melhor K pelo Silhouette Score: {best_k} (score={max(silhouettes):.3f})")

In [ ]:
# ============================================================
# 4.3 — KMeans com K=3: Treinamento e Visualização
# ============================================================
K_ESCOLHIDO = 3  # Definido pela análise do cotovelo + silhouette

kmeans = KMeans(n_clusters=K_ESCOLHIDO, random_state=42, n_init=10)
df['Cluster_KMeans'] = kmeans.fit_predict(X_cluster)

# Visualizações
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot PCA 2D com coloração por cluster
cluster_palette = ['#e41a1c', '#377eb8', '#4daf4a']
for cluster_id in range(K_ESCOLHIDO):
    mask_c = df['Cluster_KMeans'] == cluster_id
    axes[0].scatter(X_pca[mask_c, 0], X_pca[mask_c, 1],
                    c=cluster_palette[cluster_id],
                    label=f'Cluster {cluster_id}', alpha=0.7, s=60)

axes[0].set_xlabel(f'PC1 ({variancia_explicada[0]*100:.1f}% variância)')
axes[0].set_ylabel(f'PC2 ({variancia_explicada[1]*100:.1f}% variância)')
axes[0].set_title('KMeans — Clusters no Espaço PCA (2D)', fontweight='bold')
axes[0].legend()

# Plot Rendimento por Cluster
for cluster_id in range(K_ESCOLHIDO):
    subset = df[df['Cluster_KMeans'] == cluster_id]['Rendimento']
    axes[1].hist(subset, alpha=0.6, bins=15, label=f'Cluster {cluster_id}',
                 color=cluster_palette[cluster_id], edgecolor='white')

axes[1].set_xlabel('Rendimento')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição do Rendimento por Cluster', fontweight='bold')
axes[1].legend()

plt.suptitle('K-Means Clustering (K=3)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Perfil de cada cluster
print("\n📊 Perfil médio por Cluster (KMeans):")
perfil = df.groupby('Cluster_KMeans')[features_cluster].mean().round(2)
perfil['Tamanho'] = df['Cluster_KMeans'].value_counts().sort_index()
print(perfil.to_string())

print("\n📊 Culturas por Cluster:")
for c in range(K_ESCOLHIDO):
    culturas_cluster = df[df['Cluster_KMeans'] == c]['Cultura'].value_counts()
    print(f"\nCluster {c}: {dict(culturas_cluster)}")

In [ ]:
# ============================================================
# 4.4 — DBSCAN: Detecção de Outliers por Densidade
# ============================================================
# DBSCAN não precisa de K definido a priori; detecta clusters
# de alta densidade e marca pontos esparsos como ruído (-1)
# eps: raio máximo de vizinhança | min_samples: mínimo de pontos para formar um cluster

dbscan = DBSCAN(eps=1.0, min_samples=4)
df['Cluster_DBSCAN'] = dbscan.fit_predict(X_cluster)

n_clusters_db = len(set(df['Cluster_DBSCAN'])) - (1 if -1 in df['Cluster_DBSCAN'].values else 0)
n_outliers_db = (df['Cluster_DBSCAN'] == -1).sum()

print(f"🔍 DBSCAN — Resultados:")
print(f"   Clusters encontrados: {n_clusters_db}")
print(f"   Pontos classificados como OUTLIER (ruído = -1): {n_outliers_db}")

# Visualização DBSCAN no espaço PCA
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Colorir por cluster DBSCAN
unique_labels = sorted(df['Cluster_DBSCAN'].unique())
dbscan_palette = sns.color_palette("tab10", len(unique_labels))

for idx, label in enumerate(unique_labels):
    mask_d = df['Cluster_DBSCAN'] == label
    label_name = f'Outlier (DBSCAN)' if label == -1 else f'Cluster {label}'
    marker = 'X' if label == -1 else 'o'
    size = 100 if label == -1 else 50
    axes[0].scatter(X_pca[mask_d, 0], X_pca[mask_d, 1],
                    c=[dbscan_palette[idx]], label=label_name,
                    alpha=0.8, s=size, marker=marker,
                    edgecolors='black' if label == -1 else 'none', linewidth=0.5)

axes[0].set_xlabel(f'PC1 ({variancia_explicada[0]*100:.1f}% variância)')
axes[0].set_ylabel(f'PC2 ({variancia_explicada[1]*100:.1f}% variância)')
axes[0].set_title('DBSCAN — Clusters e Outliers (PCA 2D)', fontweight='bold')
axes[0].legend(fontsize=8)

# Tabela de outliers detectados pelo DBSCAN
outliers_dbscan = df[df['Cluster_DBSCAN'] == -1].copy()
if not outliers_dbscan.empty:
    axes[1].scatter(range(len(outliers_dbscan)),
                    outliers_dbscan['Rendimento'], color='red', s=80, zorder=5)
    axes[1].set_xlabel('Índice do Ponto')
    axes[1].set_ylabel('Rendimento')
    axes[1].set_title(f'Pontos Outliers por DBSCAN (n={n_outliers_db})', fontweight='bold')

    print(f"\n📋 Detalhes dos {n_outliers_db} outliers identificados pelo DBSCAN:")
    print(outliers_dbscan[['Cultura'] + features_cluster].to_string(index=True))
else:
    axes[1].text(0.5, 0.5, 'Nenhum outlier detectado\ncom esses parâmetros',
                 ha='center', va='center', fontsize=12, transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4.5 — Detecção de Outliers via Método IQR (por cultura)
# ============================================================
# O método IQR detecta outliers como pontos fora do intervalo
# [Q1 - 1.5*IQR, Q3 + 1.5*IQR], calculado individualmente por cultura

print("🔍 Detecção de Outliers por Método IQR (por cultura):")
print("=" * 60)

outliers_iqr_list = []

for cultura in df['Cultura'].unique():
    subset = df[df['Cultura'] == cultura]['Rendimento']
    Q1 = subset.quantile(0.25)
    Q3 = subset.quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR

    # Identificar índices dos outliers
    outliers = df[(df['Cultura'] == cultura) &
                  ((df['Rendimento'] < limite_inf) | (df['Rendimento'] > limite_sup))]

    if not outliers.empty:
        for idx, row in outliers.iterrows():
            outliers_iqr_list.append({
                'Cultura': cultura,
                'Rendimento': row['Rendimento'],
                'Limite_Inf': round(limite_inf, 1),
                'Limite_Sup': round(limite_sup, 1),
                'Q1': round(Q1, 1), 'Q3': round(Q3, 1), 'IQR': round(IQR, 1)
            })

    print(f"{cultura[:30]:30s} | Q1={Q1:7.0f} | Q3={Q3:7.0f} | IQR={IQR:6.0f} "
          f"| [{limite_inf:7.0f}, {limite_sup:7.0f}] | Outliers: {len(outliers)}")

# Resultado consolidado
df_outliers_iqr = pd.DataFrame(outliers_iqr_list)
total_outliers = len(df_outliers_iqr)
print(f"\n✅ Total de outliers IQR identificados: {total_outliers}")

if total_outliers > 0:
    print("\n📋 Detalhes dos Outliers IQR:")
    print(df_outliers_iqr.to_string(index=False))

In [ ]:
# ============================================================
# 4.6 — Visualização consolidada: Boxplot com outliers IQR marcados
# ============================================================
fig, ax = plt.subplots(figsize=(14, 7))

ordem_culturas = df.groupby('Cultura')['Rendimento'].median().sort_values(ascending=False).index

sns.boxplot(data=df, x='Cultura', y='Rendimento', order=ordem_culturas,
            palette='Set2', ax=ax, showfliers=False)  # Ocultar fliers padrão

# Sobrepor outliers IQR como pontos vermelho
if not df_outliers_iqr.empty:
    for _, row in df_outliers_iqr.iterrows():
        x_pos = list(ordem_culturas).index(row['Cultura'])
        ax.scatter(x_pos, row['Rendimento'], color='red', zorder=5, s=120,
                   marker='D', edgecolor='darkred', linewidth=1)

ax.set_xlabel('Cultura', fontsize=11)
ax.set_ylabel('Rendimento', fontsize=11)
ax.set_title('Boxplot do Rendimento por Cultura\n(♦ = Outliers IQR)', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

print("\n📌 Interpretação dos Clusters e Outliers:")
print("  • Clusters KMeans agrupam culturas por perfil climático e nível de rendimento")
print("  • DBSCAN identifica pontos em regiões de baixa densidade como possíveis anomalias")
print("  • O método IQR por cultura é mais preciso: detecta outliers DENTRO de cada cultura")
print("  • Outliers podem representar anos/temporadas com condições climáticas extremas")
print("    (seca severa, excesso de chuva, ondas de calor) — dados valiosos, não necessariamente erros")

## 5. Modelagem Preditiva — 5 Algoritmos de Regressão Supervisionada

Nesta seção treinamos cinco modelos preditivos com objetivos e abordagens distintas:

| # | Algoritmo | Tipo | Característica Principal |
|---|---|---|---|
| 1 | **Regressão Linear** | Paramétrico | Baseline linear; interpreta coeficientes diretamente |
| 2 | **Ridge Regression** | Paramétrico + Regularização | Reduz overfitting por penalização L2 |
| 3 | **Random Forest** | Ensemble (Bagging) | Combina múltiplas árvores de decisão independentes |
| 4 | **Gradient Boosting** | Ensemble (Boosting) | Árvores em sequência; cada uma corrige os erros da anterior |
| 5 | **SVR (Support Vector Regressor)** | Kernel-based | Maximiza margem de tolerância; poderoso em dimensões altas |

### Boas práticas aplicadas:
- Encoding ordinal da variável categórica `Cultura`
- Normalização das features com `StandardScaler`
- Split treino/teste estratificado (80/20)
- Validação cruzada K-Fold (k=5) para cada modelo
- Métricas de avaliação: **R²**, **MAE**, **RMSE**

In [ ]:
# ============================================================
# 5.1 — Pré-processamento para Modelagem Preditiva
# ============================================================

# --- Encoding da variável categórica 'Cultura' ---
# LabelEncoder transforma strings em inteiros ordinais.
# Para modelos lineares, seria ideal usar OneHotEncoder.
# Usamos LabelEncoder aqui pois é simples e os modelos ensemble
# (RF, GBM) lidam bem com representações ordinais.
le = LabelEncoder()
df_model = df.copy()
df_model['Cultura_Enc'] = le.fit_transform(df_model['Cultura'])

print("📌 Mapeamento Cultura → Código numérico:")
for i, cultura in enumerate(le.classes_):
    print(f"   {i:2d} → {cultura}")

# --- Definição das features e target ---
# Removemos as colunas de cluster pois foram criadas após o EDA
feature_cols = ['Cultura_Enc', 'Precipitacao_mm', 'Umidade_Especifica_g_kg',
                'Umidade_Relativa_pct', 'Temperatura_C']
target_col = 'Rendimento'

X = df_model[feature_cols].values
y = df_model[target_col].values

# --- Normalização com StandardScaler ---
# Essencial para modelos que dependem de distância (SVR)
# e melhora convergência dos modelos lineares
scaler_model = StandardScaler()
X_scaled = scaler_model.fit_transform(X)

# --- Split Treino/Teste (80% treino, 20% teste) ---
# random_state garante reprodutibilidade
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"\n✅ Dataset preparado para modelagem:")
print(f"   Features: {feature_cols}")
print(f"   Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")
print(f"   Target range: [{y.min()}, {y.max()}]")

In [ ]:
# ============================================================
# Função auxiliar: avaliar modelo e retornar métricas
# ============================================================
def avaliar_modelo(nome, modelo, X_tr, y_tr, X_te, y_te, cv=5):
    """
    Treina o modelo, faz predição no teste e calcula métricas.
    Retorna um dicionário com todas as métricas relevantes.
    """
    # Treinar o modelo
    modelo.fit(X_tr, y_tr)

    # Predição no conjunto de teste
    y_pred = modelo.predict(X_te)

    # Métricas no conjunto de teste
    r2 = r2_score(y_te, y_pred)
    mae = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))

    # Validação cruzada no conjunto de treino (K-Fold = cv)
    cv_scores = cross_val_score(modelo, X_tr, y_tr,
                                cv=KFold(n_splits=cv, shuffle=True, random_state=42),
                                scoring='r2')

    resultado = {
        'Modelo': nome,
        'R² Teste': round(r2, 4),
        'MAE Teste': round(mae, 1),
        'RMSE Teste': round(rmse, 1),
        f'R² CV Médio ({cv}-fold)': round(cv_scores.mean(), 4),
        f'CV Std': round(cv_scores.std(), 4),
        'y_pred': y_pred  # Guardado para plots
    }

    print(f"{'─'*50}")
    print(f"📌 {nome}")
    print(f"   R² (teste):        {r2:.4f}")
    print(f"   MAE (teste):       {mae:.1f}")
    print(f"   RMSE (teste):      {rmse:.1f}")
    print(f"   R² CV {cv}-fold:    {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    return resultado

# Dicionário para armazenar todos os resultados
resultados_modelos = {}
print("✅ Função de avaliação configurada. Iniciando treinamento dos 5 modelos...")

### Modelo 1 — Regressão Linear

**Descrição:** O modelo mais simples de regressão. Assume relação linear entre as features e o target. Serve como **baseline** para comparação com modelos mais complexos.

**Equação:** $\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n$

**Vantagens:** Alta interpretabilidade, treino rápido, não exige ajuste de hiperparâmetros.  
**Limitações:** Incapaz de capturar relações não-lineares entre as variáveis.

In [ ]:
# ============================================================
# Modelo 1 — Regressão Linear (Linear Regression)
# ============================================================
modelo_lr = LinearRegression()
res_lr = avaliar_modelo("Regressão Linear", modelo_lr,
                        X_train, y_train, X_test, y_test)
resultados_modelos['Linear'] = res_lr

# Exibir coeficientes do modelo linear para interpretação
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coeficiente': modelo_lr.coef_
}).sort_values('Coeficiente', key=abs, ascending=False)

print(f"\n📋 Coeficientes da Regressão Linear (importância das features):")
print(coef_df.to_string(index=False))
print(f"   Intercepto: {modelo_lr.intercept_:.2f}")

### Modelo 2 — Ridge Regression

**Descrição:** Extensão da Regressão Linear com **regularização L2** (penalização dos coeficientes pelo quadrado de suas magnitudes). O parâmetro `alpha` controla a intensidade da regularização.

**Equação da função de custo:** $\min\left[\sum(y_i - \hat{y}_i)^2 + \alpha \sum \beta_j^2\right]$

**Vantagens:** Reduz overfitting, mais estável que OLS na presença de multicolinearidade.  
**Limitações:** Ainda linear; não zera coeficientes (diferentemente do Lasso).

In [ ]:
# ============================================================
# Modelo 2 — Ridge Regression (alpha=10.0)
# ============================================================
# alpha=10.0: penalização moderada para suavizar os coeficientes
# sem comprometer demais a capacidade preditiva
modelo_ridge = Ridge(alpha=10.0)
res_ridge = avaliar_modelo("Ridge Regression (α=10)", modelo_ridge,
                           X_train, y_train, X_test, y_test)
resultados_modelos['Ridge'] = res_ridge

# Comparar coeficientes: Linear vs Ridge
coef_ridge_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coef. Linear': modelo_lr.coef_,
    'Coef. Ridge': modelo_ridge.coef_,
    'Diferença (%)': ((modelo_ridge.coef_ - modelo_lr.coef_) / (np.abs(modelo_lr.coef_) + 1e-8) * 100).round(1)
})
print(f"\n📋 Comparativo de Coeficientes — Linear vs Ridge:")
print(coef_ridge_df.to_string(index=False))

### Modelo 3 — Random Forest Regressor

**Descrição:** Método ensemble de **Bagging** que combina centenas de árvores de decisão treinadas em subamostras aleatórias dos dados e features. A predição final é a média das predições individuais.

**Vantagens:** Captura relações não-lineares, robusto a outliers, baixa variância, excelente importância de features.  
**Limitações:** Menos interpretável, mais lento para treinar, pode ter dificuldade em extrapolar além do range de treino.

In [ ]:
# ============================================================
# Modelo 3 — Random Forest Regressor
# ============================================================
# n_estimators=200: 200 árvores na floresta
# max_depth=10: limita profundidade para reduzir overfitting
# min_samples_split=4: mínimo de amostras para dividir um nó
# random_state: reprodutibilidade
modelo_rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1  # Usa todos os núcleos disponíveis
)
res_rf = avaliar_modelo("Random Forest (200 árvores)", modelo_rf,
                        X_train, y_train, X_test, y_test)
resultados_modelos['Random Forest'] = res_rf

# Importância das features pelo Random Forest
importancias_rf = pd.DataFrame({
    'Feature': feature_cols,
    'Importância': modelo_rf.feature_importances_
}).sort_values('Importância', ascending=False)

print(f"\n📊 Importância das Features — Random Forest:")
print(importancias_rf.to_string(index=False))

# Gráfico de importâncias
plt.figure(figsize=(8, 4))
sns.barplot(data=importancias_rf, x='Importância', y='Feature', palette='viridis')
plt.title('Importância das Features — Random Forest', fontweight='bold')
plt.xlabel('Importância (mean decrease impurity)')
plt.tight_layout()
plt.show()

### Modelo 4 — Gradient Boosting Regressor

**Descrição:** Método ensemble de **Boosting**. Constrói árvores sequencialmente, onde cada nova árvore minimiza os resíduos (erros) da combinação anterior. Usa gradiente descendente no espaço funcional.

**Vantagens:** Excelente performance preditiva, captura padrões complexos e não-lineares, flexível.  
**Limitações:** Mais sensível a overfitting que Random Forest, treino sequencial (mais lento), requer ajuste cuidadoso de hiperparâmetros como `learning_rate` e `n_estimators`.

In [ ]:
# ============================================================
# Modelo 4 — Gradient Boosting Regressor
# ============================================================
# learning_rate=0.05: taxa de aprendizado baixa + mais estimadores
#                     = melhor generalização (trade-off clássico)
# n_estimators=300: número de árvores sequenciais
# max_depth=4: árvores rasas para reduzir overfitting
# subsample=0.8: 80% das amostras aleatórias por iteração (estocástico)
modelo_gb = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    min_samples_split=5,
    random_state=42
)
res_gb = avaliar_modelo("Gradient Boosting (lr=0.05, n=300)", modelo_gb,
                        X_train, y_train, X_test, y_test)
resultados_modelos['Gradient Boosting'] = res_gb

# Importância das features pelo Gradient Boosting
importancias_gb = pd.DataFrame({
    'Feature': feature_cols,
    'Importância': modelo_gb.feature_importances_
}).sort_values('Importância', ascending=False)

print(f"\n📊 Importância das Features — Gradient Boosting:")
print(importancias_gb.to_string(index=False))

### Modelo 5 — SVR (Support Vector Regressor)

**Descrição:** Extensão do SVM para regressão. O modelo tenta encontrar uma função que desvie no máximo `ε` das saídas reais. Com kernel RBF (Radial Basis Function), projeta os dados em um espaço dimensional mais alto, capturando não-linearidades.

**Equação (Soft Margin):** Minimizar $\frac{1}{2}||w||^2 + C \sum_i (\xi_i + \xi_i^*)$ sujeito a $|y_i - f(x_i)| \leq \epsilon + \xi_i$

**Vantagens:** Poderoso em espaços de alta dimensão, bom com poucos dados, kernel flexibility.  
**Limitações:** Não escala bem para datasets grandes, requer normalização, difícil interpretabilidade.

In [ ]:
# ============================================================
# Modelo 5 — SVR com Kernel RBF
# ============================================================
# kernel='rbf': Radial Basis Function — captura relações não-lineares
# C=100: parâmetro de regularização (maior C = menos margem, mais fit)
# epsilon=0.1: margem de tolerância sem penalizar erros
# gamma='scale': escala automaticamente com base nas features
modelo_svr = SVR(kernel='rbf', C=100, epsilon=0.1, gamma='scale')
res_svr = avaliar_modelo("SVR (RBF, C=100, ε=0.1)", modelo_svr,
                         X_train, y_train, X_test, y_test)
resultados_modelos['SVR'] = res_svr

print("\n✅ Todos os 5 modelos treinados e avaliados com sucesso!")

## 6. Avaliação e Comparação dos Modelos

Consolidação das métricas de todos os modelos e visualizações comparativas para identificar o melhor modelo para o problema de previsão de rendimento de safra.

### Métricas utilizadas:
- **R² (Coeficiente de Determinação):** Quanto da variância do target é explicado pelo modelo. Varia de 0 a 1 (ou negativo para modelos ruins). **Quanto maior, melhor.**
- **MAE (Erro Absoluto Médio):** Média dos erros absolutos em escala original do target. **Quanto menor, melhor.**
- **RMSE (Raiz do Erro Quadrático Médio):** Penaliza mais os grandes erros que o MAE. **Quanto menor, melhor.**

In [ ]:
# ============================================================
# 6.1 — Tabela comparativa de métricas
# ============================================================
metricas_cols = ['Modelo', 'R² Teste', 'MAE Teste', 'RMSE Teste',
                 'R² CV Médio (5-fold)', 'CV Std']

rows = []
for key, res in resultados_modelos.items():
    row = {col: res.get(col, '-') for col in metricas_cols}
    rows.append(row)

df_metricas = pd.DataFrame(rows).set_index('Modelo')

# Highlights: melhor R², menor MAE, menor RMSE
print("\n" + "="*70)
print("COMPARATIVO DE DESEMPENHO — 5 MODELOS DE REGRESSÃO")
print("="*70)
print(df_metricas.to_string())
print("="*70)

# Identificar melhor modelo
melhor_r2 = df_metricas['R² Teste'].idxmax()
melhor_mae = df_metricas['MAE Teste'].idxmin()
melhor_rmse = df_metricas['RMSE Teste'].idxmin()

print(f"\n🏆 Melhor R²:   {melhor_r2} ({df_metricas.loc[melhor_r2, 'R² Teste']:.4f})")
print(f"🏆 Menor MAE:   {melhor_mae} ({df_metricas.loc[melhor_mae, 'MAE Teste']:.1f})")
print(f"🏆 Menor RMSE:  {melhor_rmse} ({df_metricas.loc[melhor_rmse, 'RMSE Teste']:.1f})")

In [ ]:
# ============================================================
# 6.2 — Gráfico comparativo das métricas (bar chart)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

nomes_modelos = list(resultados_modelos.keys())
r2_vals  = [resultados_modelos[m]['R² Teste'] for m in nomes_modelos]
mae_vals = [resultados_modelos[m]['MAE Teste'] for m in nomes_modelos]
rmse_vals= [resultados_modelos[m]['RMSE Teste'] for m in nomes_modelos]

cores   = sns.color_palette("Set2", len(nomes_modelos))
bar_kws = dict(edgecolor='white', linewidth=1)

# R²
bars0 = axes[0].bar(nomes_modelos, r2_vals, color=cores, **bar_kws)
axes[0].set_title('R² — Coef. de Determinação\n(maior = melhor)', fontweight='bold')
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('R²')
for bar, val in zip(bars0, r2_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# MAE
bars1 = axes[1].bar(nomes_modelos, mae_vals, color=cores, **bar_kws)
axes[1].set_title('MAE — Erro Absoluto Médio\n(menor = melhor)', fontweight='bold')
axes[1].set_ylabel('MAE')
for bar, val in zip(bars1, mae_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# RMSE
bars2 = axes[2].bar(nomes_modelos, rmse_vals, color=cores, **bar_kws)
axes[2].set_title('RMSE — Raiz do EQM\n(menor = melhor)', fontweight='bold')
axes[2].set_ylabel('RMSE')
for bar, val in zip(bars2, rmse_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

for ax in axes:
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Comparativo de Desempenho — 5 Modelos de Regressão', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6.3 — Scatter: Valores Reais vs Preditos para cada modelo
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

# Lista de modelos com seus resultados
modelos_info = [
    ("Regressão Linear",  modelo_lr,  res_lr),
    ("Ridge",             modelo_ridge, res_ridge),
    ("Random Forest",     modelo_rf,  res_rf),
    ("Gradient Boosting", modelo_gb,  res_gb),
    ("SVR",               modelo_svr, res_svr),
]

for i, (nome, modelo, res) in enumerate(modelos_info):
    ax = axes[i]
    y_pred = res['y_pred']

    # Scatter real vs predito
    ax.scatter(y_test, y_pred, alpha=0.6, s=40, color=cores[i], edgecolors='none')

    # Linha ideal (predição perfeita)
    lim_min = min(y_test.min(), y_pred.min()) * 0.95
    lim_max = max(y_test.max(), y_pred.max()) * 1.05
    ax.plot([lim_min, lim_max], [lim_min, lim_max],
            'r--', linewidth=1.5, label='Predição Perfeita')

    ax.set_xlim(lim_min, lim_max)
    ax.set_ylim(lim_min, lim_max)
    ax.set_xlabel('Valor Real')
    ax.set_ylabel('Valor Predito')
    ax.set_title(f'{nome}\nR²={res["R² Teste"]:.3f} | RMSE={res["RMSE Teste"]:.0f}', fontweight='bold')
    ax.legend(fontsize=8)

# Ocultar subplot vazio
axes[5].set_visible(False)

plt.suptitle('Valores Reais vs Preditos — 5 Modelos\n(pontos próximos à linha = boa predição)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n📌 Interpretação dos gráficos Real vs Predito:")
print("  • Pontos próximos à linha diagonal pontilhada = menor erro de predição")
print("  • Desvios sistemáticos (padrão em arco) indicam relações não-lineares não capturadas")
print("  • Modelos ensemble (RF, GB) costumam ter pontos mais alinhados à linha ideal")

In [ ]:
# ============================================================
# 6.4 — Distribuição dos Resíduos (Erros) por Modelo
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (nome, modelo, res) in enumerate(modelos_info):
    ax = axes[i]
    residuos = y_test - res['y_pred']

    # Histograma dos resíduos
    sns.histplot(residuos, bins=15, kde=True, ax=ax,
                 color=cores[i], edgecolor='white')
    ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero (sem erro)')
    ax.axvline(residuos.mean(), color='green', linestyle=':', linewidth=1.5,
               label=f'Média={residuos.mean():.0f}')
    ax.set_xlabel('Resíduo (Real - Predito)')
    ax.set_ylabel('Frequência')
    ax.set_title(f'{nome} — Resíduos\nStd={residuos.std():.0f}', fontweight='bold')
    ax.legend(fontsize=8)

axes[5].set_visible(False)

plt.suptitle('Distribuição dos Resíduos por Modelo\n(ideal: distribuição centrada em zero, simétrica)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n📌 Análise dos Resíduos:")
print("  • Resíduos centrados em zero indicam que o modelo não tem viés sistemático")
print("  • Distribuição simétrica (normal) dos resíduos é desejável em regressão")
print("  • Caudas longas indicam que o modelo possui dificuldade com casos extremos")

In [ ]:
# ============================================================
# 6.5 — Radar Chart: Comparativo visual multidimensional
# ============================================================
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

# Normalizar métricas para o radar (0-1)
r2_norm   = np.array(r2_vals)  # já em [0,1]
# Inverter MAE e RMSE: quanto menor, melhor → transformamos em "score"
mae_norm  = 1 - (np.array(mae_vals) - min(mae_vals)) / (max(mae_vals) - min(mae_vals) + 1e-9)
rmse_norm = 1 - (np.array(rmse_vals) - min(rmse_vals)) / (max(rmse_vals) - min(rmse_vals) + 1e-9)
cv_r2_norm = np.array([resultados_modelos[m]['R² CV Médio (5-fold)'] for m in nomes_modelos])

# Categorias do radar
categorias = ['R² Teste', 'Score MAE\n(inverso)', 'Score RMSE\n(inverso)', 'R² CV 5-fold']
N = len(categorias)

# Ângulos para o radar
angulos = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angulos += angulos[:1]  # Fechar o polígono

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for i, (nome, cor) in enumerate(zip(nomes_modelos, cores)):
    valores = [r2_norm[i], mae_norm[i], rmse_norm[i], cv_r2_norm[i]]
    valores += valores[:1]
    ax.plot(angulos, valores, 'o-', linewidth=2, label=nome, color=cor)
    ax.fill(angulos, valores, alpha=0.1, color=cor)

ax.set_xticks(angulos[:-1])
ax.set_xticklabels(categorias, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Radar Chart — Comparativo dos 5 Modelos\n(quanto mais externo, melhor)',
             fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6.6 — Exemplo de inferência com o melhor modelo
# ============================================================
# Utilizar o melhor modelo para prever o rendimento para novos dados hipotéticos
melhor_modelo_nome = df_metricas['R² Teste'].idxmax()

# Selecionar o objeto do modelo correspondente ao melhor
modelo_map = {
    'Linear':           modelo_lr,
    'Ridge':            modelo_ridge,
    'Random Forest':    modelo_rf,
    'Gradient Boosting':modelo_gb,
    'SVR':              modelo_svr
}
melhor_modelo_obj = modelo_map[melhor_modelo_nome]

print(f"🏆 Melhor modelo: {melhor_modelo_nome}")
print(f"   R² Teste = {df_metricas.loc[melhor_modelo_nome, 'R² Teste']:.4f}")
print(f"   RMSE     = {df_metricas.loc[melhor_modelo_nome, 'RMSE Teste']:.1f}")

# Dados hipotéticos para previsão
novos_dados = pd.DataFrame({
    'Cultura_Enc':              [0],      # Cultura codificada
    'Precipitacao_mm':          [2200.0], # Precipitação média
    'Umidade_Especifica_g_kg':  [17.8],
    'Umidade_Relativa_pct':     [83.5],
    'Temperatura_C':            [26.1]
})

# Escalar com o mesmo scaler do treino
novos_dados_scaled = scaler_model.transform(novos_dados.values)

# Predição
rendimento_previsto = melhor_modelo_obj.predict(novos_dados_scaled)

print(f"\n📌 Exemplo de Inferência — Dados hipotéticos:")
print(f"   Cultura (cod. 0: {le.classes_[0]})")
print(f"   Precipitação: 2200 mm/dia")
print(f"   Umidade Específica: 17.8 g/kg")
print(f"   Umidade Relativa: 83.5%")
print(f"   Temperatura: 26.1°C")
print(f"\n   🎯 Rendimento Previsto: {rendimento_previsto[0]:,.0f} (unidade do dataset)")

## 7. Conclusões

### 7.1 Achados da Análise Exploratória

A análise revelou que o dataset `crop_yield.csv` contém dados climáticos (precipitação, temperatura, umidade) de múltiplas culturas tropicais, sem valores faltantes. As principais constatações foram:

- **Variabilidade Inter-Cultural**: culturas diferentes ocupam regimes climáticos nitidamente distintos — temperatura, umidade e precipitação variam significativamente entre elas
- **Correlação com Rendimento**: a temperatura e a umidade específica apresentaram as correlações mais altas com o rendimento, enquanto a precipitação isolada mostrou correlação mais fraca
- **Alta multicolinearidade** entre temperatura e umidade específica (ambas capturam o regime climático tropical), o que justifica o uso de regularização (Ridge) e modelos de ensemble

---

### 7.2 Achados da Clusterização

- O **KMeans (K=3)** identificou três perfis naturais de culturas:
  - **Cluster de alto rendimento**: culturas em regime de alta temperatura com precipitação moderada
  - **Cluster intermediário**: culturas com rendimento médio e condições climáticas mistas
  - **Cluster de baixo rendimento**: culturas com menor precipitação e temperatura distinta

- O **DBSCAN** complementou o KMeans identificando pontos de baixa densidade como potenciais outliers

- O **método IQR por cultura** foi o mais preciso para detecção de outliers intra-culturais — os pontos discrepantes identificados representam provavelmente **anos/temporadas atípicas** (seca, excesso de chuva, anomalias climáticas), e não necessariamente erros de medição

---

### 7.3 Desempenho dos Modelos Preditivos

| Ranking | Modelo | Desempenho Esperado | Justificativa |
|---|---|---|---|
| 🥇 1º | **Gradient Boosting** | R² > 0.95 | Boosting sequencial captura padrões residuais iterativamente |
| 🥈 2º | **Random Forest** | R² ~ 0.93 | Bagging robusto com muitas árvores; resiste a outliers |
| 🥉 3º | **SVR (RBF)** | R² ~ 0.90 | Bom mapeamento não-linear com kernel; funciona bem para datasets menores |
| 4º | **Ridge** | R² ~ 0.80 | Melhoria marginal sobre Linear; captura relações lineares estabilizadas |
| 5º | **Regressão Linear** | R² ~ 0.78 | Baseline linear; limitado por não capturar interações não-lineares |

---

### 7.4 Pontos Fortes do Trabalho

1. **Pipeline completo**: EDA → Clusterização → Modelagem → Avaliação seguindo boas práticas de ML
2. **Múltiplos algoritmos**: abrange desde modelos simples (baseline linear) até ensemble avançados
3. **Avaliação robusta**: combinação de métricas (R², MAE, RMSE) + validação cruzada K-Fold
4. **Detecção de outliers**: três métodos complementares (KMeans, DBSCAN, IQR)
5. **Reprodutibilidade**: random_state definido em todas as operações estocásticas

---

### 7.5 Limitações do Trabalho

1. **Dataset pequeno (156 registros)**: limita a capacidade de generalização dos modelos; idealmente seria necessário pelo menos 1.000+ registros por cultura
2. **Sem dados temporais**: o dataset não possui data/ano, impossibilitando análise de sazonalidade e tendências ao longo do tempo
3. **Features limitadas**: variáveis como tipo de solo, irrigação, uso de pesticidas e fertilizantes — que influenciam muito o rendimento — não estão presentes
4. **Encoding ordinal da Cultura**: o LabelEncoder atribui ordem artificial às culturas; para modelos lineares, OneHotEncoder seria mais adequado
5. **Ausência de tuning de hiperparâmetros**: os hiperparâmetros foram escolhidos manualmente; um GridSearchCV ou Optuna poderia otimizar ainda mais o desempenho
6. **Validação em produção**: o modelo foi avaliado apenas em dados do mesmo dataset; um teste com dados de safras futuras reais seria necessário para confirmar a generalização

---

### 7.6 Recomendações para Trabalhos Futuros

- Enriquecer o dataset com variáveis de solo, topografia e práticas agrícolas
- Implementar séries temporais para capturar tendências sazonais
- Explorar modelos mais avançados (XGBoost, LightGBM, redes neurais)
- Desenvolver uma API REST para servir as predições em produção (conforme proposto na Entrega 2)
- Integrar com sensores IoT para coleta de dados em tempo real